Imlo coursework

In [12]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [13]:
# code to use my gpu
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

cuda


In [14]:
#transform to the train dataset with augmentation
train_transform = transforms.Compose([
    # adding random augmentations
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# transforms for the val dataset without data augmentations
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [15]:
#defining the train dataset
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = train_transform,
    download = True
)

#defining the val dataset
val_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = val_transform,
    download = False
)

#split will be 80:20
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size


train_indices, val_indices = torch.utils.data.random_split(
    range(len(train_data)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

train_data = torch.utils.data.Subset(
    train_data,
    train_indices.indices
)

val_data = torch.utils.data.Subset(
    val_data,
    val_indices.indices
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)

In [16]:
image, label = train_data[0]

In [17]:
image.size()

torch.Size([3, 128, 128])

In [18]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [19]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5)
        self.conv2 = nn.Conv2d(12, 24, 5)

        self.conv3 = nn.Conv2d(24, 24, 5)
        self.conv4 = nn.Conv2d(24, 24, 5)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.dropout = nn.Dropout(0.2)

        self.fc1 = nn.Linear(24 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 37)

    def forward(self, input):
        input = F.relu(self.conv1(input)) #conv1 then relu
        input = F.relu(self.conv2(input)) #conv2 then relu
        input = self.pool(input) #max pool
        input = F.relu(self.conv3(input)) #conv3 then relu
        input = F.relu(self.conv4(input)) #conv4 then relu
        input = self.pool(input) #max pool
        input = self.adaptive_pool(input) #adaptive pool

        input = torch.flatten(input, 1)  #flattening
        input = F.relu(self.fc1(input))  #applying fc1, then RELU
        input = F.relu(self.fc2(input))  #applying fc2, then RELU
        input = self.dropout(input)  #applying dropout
        input = self.fc3(input)  #applying fc3
        return input

In [20]:
# defining the NN itself
network = NeuralNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.001)

In [21]:
# training the model
for epoch in range(30):
    print("Training epoch:", epoch)
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = network(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

    running_loss_calc = running_loss / len(train_loader)
    print("Loss:", running_loss_calc)

Training epoch: 0
Loss: 3.6122397495352705
Training epoch: 1
Loss: 3.5241801116777505
Training epoch: 2
Loss: 3.428263280702674
Training epoch: 3
Loss: 3.39616917008939
Training epoch: 4
Loss: 3.357229103212771
Training epoch: 5
Loss: 3.3269799999568774
Training epoch: 6
Loss: 3.29716171907342
Training epoch: 7
Loss: 3.2347758230955703
Training epoch: 8
Loss: 3.1975309174993765
Training epoch: 9
Loss: 3.142905463343081
Training epoch: 10
Loss: 3.118411851965863
Training epoch: 11
Loss: 3.090287659479224
Training epoch: 12
Loss: 3.0616951144259907
Training epoch: 13
Loss: 3.034405283305956
Training epoch: 14
Loss: 2.9958030037257983
Training epoch: 15
Loss: 2.950183251629705
Training epoch: 16
Loss: 2.8983765581379766
Training epoch: 17
Loss: 2.8778259080389272
Training epoch: 18
Loss: 2.8505370046781455
Training epoch: 19
Loss: 2.8162620845048325
Training epoch: 20
Loss: 2.816944428112196
Training epoch: 21
Loss: 2.7349754209103794
Training epoch: 22
Loss: 2.721919806107231
Training ep

In [22]:
# testing the model on val data
correct = 0
total = 0

network.eval()

with torch.no_grad():
  for images, labels in val_loader:

    images = images.to(device)
    labels = labels.to(device)


    outputs = network(images)
    predicted = outputs.argmax(1)

    total += len(labels)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 18.070652173913043
